# Parameterised virtual products ("knobs")

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

A virtual product callback can declare **knobs** — runtime-tunable parameters that show up in the inspector panel (and as widgets next to a `%%vp --debug` cell). Change a knob → SciQLop re-fetches → the graph updates.

Any keyword argument with a default value (beyond `start` and `stop`) automatically becomes a knob. Add `Annotated[T, Knob(...)]` for richer metadata (bounds, step, label, unit, widget hints).

**What you'll learn**
- Add simple float / int / bool knobs by giving your callback keyword arguments with defaults.
- Add rich metadata (bounds, step, units, custom labels) with `Annotated[..., Knob(...)]`.
- Use `Literal` for dropdown choices.
- Use **visual knobs** — draggable horizontal lines (`widget="hline"`) and draggable time spans (`SciQLopPlotRange`) — that the user manipulates directly on the plot.
- Build worked examples (mirror threshold, MVA) that combine knobs with real MMS data.

**Prerequisites** — [Virtual products](6-SciQLopVirtualProducts.ipynb).

**Next** — [DSP](14-SciQLopDSP.ipynb) (knobs in DSP pipelines), [Annotation layers](16-SciQLopAnnotationLayers.ipynb) (knobs in layers).


## 1. Basic knobs — just add keyword arguments

The simplest way to add knobs: give your callback extra keyword arguments with defaults.
SciQLop will detect them and create UI controls in the inspector.

In [ ]:
%%vp --path "examples/tunable_sine" --debug --start "2020-01-01" --stop "2020-01-02"
import numpy as np

def tunable_sine(start: float, stop: float,
                 frequency: float = 0.01,
                 amplitude: float = 1.0,
                 offset: float = 0.0) -> Scalar["signal"]:
    t = np.arange(start, stop, 5.0)
    y = offset + amplitude * np.sin(2 * np.pi * frequency * (t - start))
    return t, y

The `--debug` flag creates a plot panel and — when `ipywidgets` is installed —
displays interactive sliders below the cell. Try changing the frequency or amplitude!

You can also tune the parameters from the **Inspector** panel in the SciQLop GUI
(look for the "Parameters" section when the graph is selected).

## 2. Rich metadata with `Annotated` and `Knob`

For finer control — bounds, step size, labels, units — use `Annotated[T, Knob(...)]`.

In [ ]:
%%vp --path "examples/filtered_noise" --debug --start "2020-01-01" --stop "2020-01-02"
import numpy as np
from typing import Annotated
from SciQLop.user_api.knobs import Knob
from scipy.signal import lfilter

def filtered_noise(
    start: float, stop: float,
    cutoff: Annotated[float, Knob(min=0.001, max=1.0, step=0.001,
                                  label="Cutoff freq", unit="Hz")] = 0.1,
    amplitude: Annotated[float, Knob(min=0.1, max=10.0, step=0.1)] = 1.0,
) -> Scalar["filtered"]:
    t = np.arange(start, stop, 1.0)
    rng = np.random.default_rng(42)
    noise = amplitude * rng.standard_normal(len(t))
    alpha = min(1.0, 2 * np.pi * cutoff)
    out = lfilter([alpha], [1, -(1 - alpha)], noise)
    return t, out

The `Knob(...)` metadata translates to:
- `min` / `max` → slider bounds (and spinbox bounds in the inspector)
- `step` → increment step
- `label` → display name in the UI (defaults to the parameter name)
- `unit` → shown next to the widget

## 3. Choice knobs with `Literal`

Use a `Literal` type annotation for enum-style choices. SciQLop renders these
as a dropdown menu.

In [ ]:
%%vp --path "examples/waveform" --debug --start "2020-01-01" --stop "2020-01-02"
import numpy as np
from typing import Literal

def waveform(start: float, stop: float,
             shape: Literal["sine", "square", "sawtooth"] = "sine",
             frequency: float = 0.01,
             ) -> Scalar["wave"]:
    t = np.arange(start, stop, 1.0)
    phase = 2 * np.pi * frequency * (t - start)
    if shape == "sine":
        y = np.sin(phase)
    elif shape == "square":
        y = np.sign(np.sin(phase))
    else:
        y = 2 * (phase / (2 * np.pi) % 1) - 1
    return t, y

## 4. Boolean knobs

A `bool` kwarg becomes a checkbox in the inspector.

In [ ]:
%%vp --path "examples/noisy_signal" --debug --start "2020-01-01" --stop "2020-01-02"
import numpy as np

def noisy_signal(start: float, stop: float,
                 add_noise: bool = True,
                 noise_level: float = 0.2,
                 ) -> Scalar["value"]:
    t = np.arange(start, stop, 1.0)
    y = np.sin((t - start) / 100)
    if add_noise:
        rng = np.random.default_rng(int(start) % 2**31)
        y = y + noise_level * rng.standard_normal(len(t))
    return t, y

## 5. Visual knobs — interactive plot controls

Some knobs work best as **interactive plot items** rather than spinboxes:

- **`ThresholdKnob`**: A horizontal line you can drag up/down. Use `Knob(widget="hline")`
  — the value is a float, rendered as a movable HLine on the plot.
- **`TimeRangeKnob`**: A vertical span you can drag/resize. Type-hint a parameter as
  `SciQLopPlotRange` — the value is a time range, rendered as a draggable VSpan that
  stays anchored to the visible window when you pan.

These visual knobs sync bidirectionally with the inspector panel.

In [ ]:
%%vp --path "examples/mirror_tunable" --debug --start "2015-11-18T02:14:30" --stop "2015-11-18T03:34:00"
import numpy as np
from typing import Annotated
from SciQLop.user_api.knobs import Knob
import speasy as spz
import scipy.constants as cst
from speasy.signal.resampling import interpolate

def mirror_tunable(
    start: float, stop: float,
    smooth_window: Annotated[int, Knob(min=1, max=100, step=1,
                                       label="Smooth window")] = 1,
    threshold: Annotated[float, Knob(widget="hline", min=0.0, max=5.0,
                                     step=0.1, label="Threshold",
                                     color="#e74c3c")] = 1.0,
) -> MultiComponent["C_mirror", "threshold"]:
    b_gse = spz.get_data("cda/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2", start, stop)
    t_para = spz.get_data("cda/MMS1_FPI_FAST_L2_DIS-MOMS/mms1_dis_temppara_fast", start, stop)
    t_perp = spz.get_data("cda/MMS1_FPI_FAST_L2_DIS-MOMS/mms1_dis_tempperp_fast", start, stop)

    if any(v is None for v in [b_gse, t_para, t_perp]):
        return None

    # interpolate(ref, var) — resample FGM onto FPI's coarser time grid
    b_gse = interpolate(t_para, b_gse)

    b_mag = np.sqrt(np.sum(b_gse.values[:, :3] ** 2, axis=1)) * 1e-9
    beta_perp = 2 * cst.mu_0 * t_perp.values.flatten() * cst.eV / (b_mag ** 2)
    ratio = t_perp.values.flatten() / t_para.values.flatten()
    c_mirror = beta_perp * (ratio - 1)

    if smooth_window > 1:
        kernel = np.ones(smooth_window) / smooth_window
        c_mirror = np.convolve(c_mirror, kernel, mode="same")

    t = b_gse.time.astype("datetime64[ns]").astype(np.int64) / 1e9
    thr = np.full_like(c_mirror, threshold)
    return t, np.column_stack([c_mirror, thr])

The `threshold` knob above uses `widget="hline"` — it renders as a draggable
horizontal line on the plot. Drag it up or down to change the instability threshold
and see the mirror criterion re-evaluated live.

### TimeRangeKnob — select a sub-interval on the plot

Use a `SciQLopPlotRange` type hint to create a draggable vertical span.
The span stays visually anchored when you pan — its data-space range is
recomputed automatically and passed to the callback.

In [ ]:
%%vp --path "examples/windowed_stats" --debug --start "2020-01-01" --stop "2020-01-02"
import numpy as np
from SciQLop.user_api.knobs import SciQLopPlotRange

def windowed_stats(
    start: float, stop: float,
    analysis_window: SciQLopPlotRange = SciQLopPlotRange(0.3, 0.7),
) -> MultiComponent["signal", "window_mean"]:
    t = np.arange(start, stop, 5.0)
    y = np.sin(2 * np.pi * 0.01 * (t - start)) + 0.3 * np.random.default_rng(42).standard_normal(len(t))
    w_start, w_stop = analysis_window.start(), analysis_window.stop()
    mask = (t >= w_start) & (t <= w_stop)
    window_mean = np.where(mask, np.nanmean(y[mask]), np.nan)
    return t, np.column_stack([y, window_mean])


### MVA — Minimum Variance Analysis with a draggable window

A classic use case for `TimeRangeKnob`: select the analysis interval for
Minimum Variance Analysis (MVA) of the magnetic field directly on the plot.

MVA finds the coordinate frame in which the magnetic field variance is
decomposed into maximum, intermediate, and minimum directions by
eigendecomposition of the covariance matrix (Sonnerup & Scheible, 1998).
The minimum variance direction estimates the normal to a current sheet or
boundary. The eigenvalue ratio λ₂/λ₃ measures the quality of the determination.

Drag the blue span to move the analysis window over different structures.

In [ ]:
%%vp --path "examples/mva" --debug --start "2015-11-18T02:14:30" --stop "2015-11-18T03:34:00"
import numpy as np
import speasy as spz
from SciQLop.user_api.knobs import SciQLopPlotRange
from speasy.signal.resampling import interpolate

def mva(
    start: float, stop: float,
    analysis_window: SciQLopPlotRange = SciQLopPlotRange(0.3, 0.7),
) -> MultiComponent["B_max", "B_int", "B_min"]:
    b_gse = spz.get_data("cda/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2", start, stop)
    if b_gse is None:
        return None

    t = b_gse.time.astype("datetime64[ns]").astype(np.int64) / 1e9
    B = b_gse.values[:, :3].astype(np.float64)

    w_start, w_stop = analysis_window.start(), analysis_window.stop()
    mask = (t >= w_start) & (t <= w_stop)
    if mask.sum() < 4:
        return t, np.full((len(t), 3), np.nan)

    B_win = B[mask]
    mean = B_win.mean(axis=0)
    cov = (B_win - mean).T @ (B_win - mean) / len(B_win)

    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    # eigh returns ascending order; reverse to λ1 ≥ λ2 ≥ λ3
    idx = np.argsort(eigenvalues)[::-1]
    eigenvectors = eigenvectors[:, idx]

    projected = B @ eigenvectors
    return t, projected


## 6. Knobs with the programmatic API

You can also create parameterized VPs outside of `%%vp` using `create_virtual_product`.

In [ ]:
import numpy as np
from typing import Annotated, Literal
from SciQLop.user_api.knobs import Knob
from SciQLop.user_api.virtual_products import create_virtual_product, VirtualProductType

def damped_oscillator(
    start: float, stop: float,
    frequency: Annotated[float, Knob(min=0.001, max=0.1, step=0.001)] = 0.01,
    damping: Annotated[float, Knob(min=0.0, max=0.01, step=0.0001)] = 0.001,
    envelope: Literal["exp", "linear", "none"] = "exp",
):
    t = np.arange(start, stop, 5.0)
    rel = t - start
    y = np.sin(2 * np.pi * frequency * rel)
    if envelope == "exp":
        y *= np.exp(-damping * rel)
    elif envelope == "linear":
        y *= np.maximum(0, 1 - damping * rel)
    return t, y

vp = create_virtual_product(
    "examples/damped_oscillator",
    damped_oscillator,
    VirtualProductType.Scalar,
    labels=["oscillation"]
)

In [ ]:
from SciQLop.user_api.plot import create_plot_panel
from SciQLop.user_api import TimeRange

p = create_plot_panel()
p.time_range = TimeRange("2020-01-01", "2020-01-02")
p.plot(vp)

## 7. Hot-reload preserves knob values

When you re-execute a `%%vp` cell:
- If only the **function body** changed (same kwargs), the callback is hot-swapped
  and knob values stay as-is.
- If you **add, remove, or rename** a kwarg, SciQLop migrates the knob values:
  existing valid values are kept, removed knobs are dropped, new knobs start at their default.

## 8. Supported knob types

| Python type | Inspector widget | Plot item | Notebook widget |
|---|---|---|---|
| `int` | `QSpinBox` | — | `IntSlider` |
| `float` | `QDoubleSpinBox` | — | `FloatSlider` |
| `bool` | `QCheckBox` | — | `Checkbox` |
| `Literal["a", "b", ...]` | `QComboBox` | — | `Dropdown` |
| `str` | `QLineEdit` | — | `Text` |
| `SciQLopPlotRange` | read-only label | draggable VSpan | — |
| `Knob(widget="hline")` | `QDoubleSpinBox` | draggable HLine | `FloatSlider` |

All types support `Annotated[T, Knob(label=, unit=, description=)]` for UI hints.

Visual knobs (`SciQLopPlotRange` and `widget="hline"`) create interactive items
directly on the plot that sync bidirectionally with the inspector.

## 9. How it works under the hood

1. When you drop a parameterized VP on a panel (or plot it from Python),
   the graph is initialized with default knob values.
2. Each knob change triggers a debounced re-fetch (~400 ms delay).
3. Knob values are included in the cache key, so different parameter
   combinations are cached independently.
4. The graph inspector shows a "Parameters" section with a widget per knob.
5. An info-badge overlay on the graph summarizes the current knob values.

## API reference

### Knob metadata

```python
from SciQLop.user_api.knobs import Knob

Knob(min=, max=, step=, label=, unit=, description=, apply=, choices=, pattern=,
     widget=, color=)
```

### Knob spec types (for advanced usage)

| Class | Fields |
|---|---|
| `IntKnob` | `default`, `min`, `max`, `step` |
| `FloatKnob` | `default`, `min`, `max`, `step` |
| `BoolKnob` | `default` |
| `ChoiceKnob` | `default`, `choices` (list of `(display_name, value)` pairs) |
| `StringKnob` | `default`, `pattern` (optional regex) |
| `TimeRangeKnob` | `default` (`SciQLopPlotRange`), `color` — draggable VSpan |
| `ThresholdKnob` | `default`, `min`, `max`, `step`, `color` — draggable HLine |

All specs share: `name`, `label`, `unit`, `description`, `apply` (`"live"` or `"manual"`), `widget`.

### Visual knobs shorthand

```python
from SciQLop.user_api.knobs import Knob, SciQLopPlotRange
from typing import Annotated

# TimeRangeKnob — use SciQLopPlotRange as the type hint:
def my_vp(start, stop,
          window: SciQLopPlotRange = SciQLopPlotRange(0.3, 0.7)):
    ...

# ThresholdKnob — use Knob(widget="hline"):
def my_vp(start, stop,
          threshold: Annotated[float, Knob(widget="hline", color="#e74c3c")] = 5.0):
    ...
```
